In [24]:
import requests
import pandas as pd
import numpy as np
from gprofiler import GProfiler
from collections import defaultdict
from typing import List, Dict, Any, Optional
import spacy
import xml.etree.ElementTree as ET
import re
import torch
import time
from tqdm import tqdm
from multiprocessing.dummy import Pool as ThreadPool
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nlp = spacy.load("en_core_web_sm")

In [3]:
API_URL = "https://api.platform.opentargets.org/api/v4/graphql"
DISEASE_ID = "MONDO_0005180"
query = f"""
query {{
  disease(efoId: "{DISEASE_ID}") {{
    associatedTargets(page: {{ index: 0, size: 100 }}) {{
      count
      rows {{
        target {{
          id
          approvedSymbol
          approvedName
        }}
        score
      }}
    }}
  }}
}}
"""
response = requests.post(API_URL, json={"query":query})
if response.status_code != 200:
    raise Exception("Something went wrong while fetching genes")
else:
    target_genes_response = response.json()
print(target_genes_response)



{'data': {'disease': {'associatedTargets': {'count': 6584, 'rows': [{'target': {'id': 'ENSG00000188906', 'approvedSymbol': 'LRRK2', 'approvedName': 'leucine rich repeat kinase 2'}, 'score': 0.8789207866590859}, {'target': {'id': 'ENSG00000145335', 'approvedSymbol': 'SNCA', 'approvedName': 'synuclein alpha'}, 'score': 0.8585012729359579}, {'target': {'id': 'ENSG00000159363', 'approvedSymbol': 'ATP13A2', 'approvedName': 'ATPase cation transporting 13A2'}, 'score': 0.8525516685262865}, {'target': {'id': 'ENSG00000185345', 'approvedSymbol': 'PRKN', 'approvedName': 'parkin RBR E3 ubiquitin protein ligase'}, 'score': 0.8505122712454569}, {'target': {'id': 'ENSG00000158828', 'approvedSymbol': 'PINK1', 'approvedName': 'PTEN induced kinase 1'}, 'score': 0.8489915009692078}, {'target': {'id': 'ENSG00000116675', 'approvedSymbol': 'DNAJC6', 'approvedName': 'DnaJ heat shock protein family (Hsp40) member C6'}, 'score': 0.813216808888733}, {'target': {'id': 'ENSG00000116288', 'approvedSymbol': 'PARK7

In [4]:
def get_target_genes(response:dict) -> dict:
    target_res = response['data']['disease']['associatedTargets']['rows']
    genes = []
    for i in target_res:
        gene_data = {
            "gene_id": i['target']['id'],
            "symbol":  i['target']['approvedSymbol'],
            "name": i['target']['approvedName'],
            "score": i['score']
        }
        genes.append(gene_data)
    return genes

genes_target = get_target_genes(target_genes_response)
print(genes_target)

[{'gene_id': 'ENSG00000188906', 'symbol': 'LRRK2', 'name': 'leucine rich repeat kinase 2', 'score': 0.8789207866590859}, {'gene_id': 'ENSG00000145335', 'symbol': 'SNCA', 'name': 'synuclein alpha', 'score': 0.8585012729359579}, {'gene_id': 'ENSG00000159363', 'symbol': 'ATP13A2', 'name': 'ATPase cation transporting 13A2', 'score': 0.8525516685262865}, {'gene_id': 'ENSG00000185345', 'symbol': 'PRKN', 'name': 'parkin RBR E3 ubiquitin protein ligase', 'score': 0.8505122712454569}, {'gene_id': 'ENSG00000158828', 'symbol': 'PINK1', 'name': 'PTEN induced kinase 1', 'score': 0.8489915009692078}, {'gene_id': 'ENSG00000116675', 'symbol': 'DNAJC6', 'name': 'DnaJ heat shock protein family (Hsp40) member C6', 'score': 0.813216808888733}, {'gene_id': 'ENSG00000116288', 'symbol': 'PARK7', 'name': 'Parkinsonism associated deglycase', 'score': 0.8106296934517689}, {'gene_id': 'ENSG00000100225', 'symbol': 'FBXO7', 'name': 'F-box protein 7', 'score': 0.8055069071068177}, {'gene_id': 'ENSG00000177628', 'sy

In [5]:
genes_target_prelim_df = pd.DataFrame(genes_target)
genes_target_prelim_df

,gene_id,symbol,name,score
0,ENSG00000188906,LRRK2,leucine rich repeat kinase 2,0.878921
1,ENSG00000145335,SNCA,synuclein alpha,0.858501
2,ENSG00000159363,ATP13A2,ATPase cation transporting 13A2,0.852552
3,ENSG00000185345,PRKN,parkin RBR E3 ubiquitin protein ligase,0.850512
4,ENSG00000158828,PINK1,PTEN induced kinase 1,0.848992
...,...,...,...,...
95,ENSG00000010810,FYN,"FYN proto-oncogene, Src family tyrosine kinase",0.497532
96,ENSG00000113712,CSNK1A1,casein kinase 1 alpha 1,0.497218
97,ENSG00000169676,DRD5,dopamine receptor D5,0.496620
98,ENSG00000164615,CAMLG,calcium modulating ligand,0.494202


In [6]:
def load_gtex_expression(filepath: str):
    df = pd.read_csv(filepath, sep="\t", skiprows=2, compression='gzip')
    brain_cols = [col for col in df.columns if 'Brain' in col]
    df["median_tpm_brain"] = df[brain_cols].median(axis=1)
    df = df.rename(columns={"Name": "gene_id", "Description": "symbol"})
    df["gene_id_stripped"] = df["gene_id"].str.split(".").str[0]
    return df[["gene_id_stripped", "symbol", "median_tpm_brain"]]

gene_tpm_brain_df = load_gtex_expression("GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm (1).gct.gz")

def merge_open_targets_gtex(df_1, df_2):
    merged_df = df_1.merge(
        df_2,
        left_on="gene_id",
        right_on="gene_id_stripped",
        how="left"
    )
    merged_df = merged_df.drop(columns=["gene_id_stripped"])
    return merged_df

mereged_genes_parkinsons = merge_open_targets_gtex(genes_target_prelim_df, gene_tpm_brain_df)


In [7]:
mereged_genes_parkinsons
def prioritize_genes(df):
    new_df = df.copy()
    new_df["median_tpm_brain"] = new_df["median_tpm_brain"].fillna(0)
    new_df["final_score"] =  np.log2(1+new_df["median_tpm_brain"]) * new_df["score"]
    new_df = new_df.sort_values(by="final_score", ascending=False)
    return new_df

final_genes_parkinsons_sorted = prioritize_genes(mereged_genes_parkinsons)


In [8]:
final_genes_parkinsons_sorted

,gene_id,symbol_x,name,score,symbol_y,median_tpm_brain,final_score
6,ENSG00000116288,PARK7,Parkinsonism associated deglycase,0.810630,PARK7,182.992000,6.098772
15,ENSG00000197746,PSAP,prosaposin,0.644604,PSAP,633.102000,6.000345
2,ENSG00000159363,ATP13A2,ATPase cation transporting 13A2,0.852552,ATP13A2,59.661500,5.049416
36,ENSG00000106153,CHCHD2,coiled-coil-helix-coiled-coil-helix domain con...,0.586215,CHCHD2,322.936000,4.888780
47,ENSG00000168653,NDUFS5,NADH:ubiquinone oxidoreductase subunit S5,0.544431,NDUFS5,428.624000,4.762104
...,...,...,...,...,...,...,...
59,ENSG00000178999,AURKB,aurora kinase B,0.526138,AURKB,0.053693,0.039700
18,ENSG00000151577,DRD3,dopamine receptor D3,0.629745,DRD3,0.011972,0.010812
99,ENSG00000094755,GABRP,gamma-aminobutyric acid type A receptor subuni...,0.491403,GABRP,0.015352,0.010801
88,ENSG00000103546,SLC6A2,solute carrier family 6 member 2,0.499819,SLC6A2,0.007738,0.005558


In [9]:
def run_gprofiler(symbols: list):
    gp = GProfiler(return_dataframe=True)
    results = gp.profile(organism='hsapiens', query=symbols, no_evidences=False)
    return results

gprofiler_res = run_gprofiler(final_genes_parkinsons_sorted.head(50)['symbol_x'].dropna().tolist())

In [10]:
gprofiler_res.to_csv('OUTPUT_CSV.csv', index=False)

In [11]:
def enrich_top_200_genes(gprofiler_top):
    enriched_score = defaultdict(float)
    for _, row in gprofiler_top.iterrows():
        if (len(row["intersections"]) == 0):
            continue
        for i in row["intersections"]:
            enriched_score[i] += -np.log10(row["p_value"])/len(row["intersections"])
    return enriched_score

enriched_score = enrich_top_200_genes(gprofiler_res)


In [12]:
enriched_score

defaultdict(float,
            {'PARK7': 121.5375532654405,
             'PSAP': 31.40551603777035,
             'ATP13A2': 65.1683109253922,
             'CHCHD2': 36.74917827530963,
             'SNCA': 126.88162502179861,
             'UCHL1': 65.59297273493974,
             'FBXO7': 42.943653448993686,
             'MAPT': 65.73791684532166,
             'EIF4G1': 54.17538887267741,
             'VPS35': 83.74497758722529,
             'UQCRC1': 50.137011537967325,
             'PTPA': 20.501792481830638,
             'GBA1': 72.90899859908062,
             'PLA2G6': 43.07518869425227,
             'HTRA2': 57.31597835436296,
             'PINK1': 118.9841456419906,
             'DNAJC6': 57.304407678769785,
             'COMT': 30.716059337324523,
             'ATP6V1A': 71.27928671946869,
             'SMPD1': 21.441393286679837,
             'NDUFS5': 80.41081352468312,
             'NDUFB7': 74.04741025603654,
             'NDUFA8': 77.96713234458757,
             'NDUFV2': 76.

In [13]:
def compute_enriched_score(df_200, g_profiler_scores_dict, alpha):
    df_200_final = df_200.copy()
    df_200_final["g_profiler_scores"] = df_200_final["symbol_x"].map(g_profiler_scores_dict).fillna(0)
    df_200_final["final_gene_score"] = df_200_final["final_score"] + alpha*df_200_final["g_profiler_scores"]
    return df_200_final.sort_values(by="final_gene_score", ascending=False)

final_genes_scored = compute_enriched_score(final_genes_parkinsons_sorted.head(50), enriched_score, 1.0)

In [14]:
final_genes_scored.head(5)

,gene_id,symbol_x,name,score,symbol_y,median_tpm_brain,final_score,g_profiler_scores,final_gene_score
1,ENSG00000145335,SNCA,synuclein alpha,0.858501,SNCA,38.2467,4.545335,126.881625,131.426960
6,ENSG00000116288,PARK7,Parkinsonism associated deglycase,0.810630,PARK7,182.9920,6.098772,121.537553,127.636325
4,ENSG00000158828,PINK1,PTEN induced kinase 1,0.848992,PINK1,41.0024,4.578102,118.984146,123.562247
9,ENSG00000069329,VPS35,VPS35 retromer complex component,0.760328,VPS35,19.3967,3.307626,83.744978,87.052604
47,ENSG00000168653,NDUFS5,NADH:ubiquinone oxidoreductase subunit S5,0.544431,NDUFS5,428.6240,4.762104,80.410814,85.172917


In [15]:
# check literature hits for the genes shortlisted in Workflow 1

def check_eu_pmc(disease: str, gene: str) -> dict:
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    try:
        response = requests.get(url, {
            "query": f'"{gene}" AND "{disease}"',
            "format": "json",
            "pageSize": 100
        })
        response.raise_for_status()
        data = response.json()
        hit_count = int(data.get("hitCount", 0))
        res = data.get("resultList", {}).get("result", [])
        years = [int(x["pubYear"]) for x in res if "pubYear" in x]
        fulltext_hits = sum(1 for paper in res if paper.get("hasTextMinedTerms") == "Y")
        return {
            "gene":gene,
            "hit_count":hit_count,
            "fulltext_hits":fulltext_hits,
            "mean_pub_year": np.mean(years) if years else np.nan
        }
    except Exception as e:
        raise RuntimeError(f"Error while fetching from EU PMC: {str(e)}")


In [16]:
def check_literature_match_for_genes(gene_list: list, disease: str) -> dict:
    return [check_eu_pmc(disease, x) for x in gene_list]

genes_lit_match = check_literature_match_for_genes(final_genes_scored["symbol_x"].to_list(), "Parkinson's Disease")
genes_lit_match_df = pd.DataFrame(genes_lit_match)

In [17]:
genes_lit_match_df.head(5)

,gene,hit_count,fulltext_hits,mean_pub_year
0,SNCA,7608,97,2024.86
1,PARK7,2316,98,2024.44
2,PINK1,6453,98,2024.75
3,VPS35,1397,100,2024.27
4,NDUFS5,50,50,2018.24


In [18]:
class LiteratureFetcher:

    def __init__(self, gene_list: List[str], disease_name: str, page_size: int = 20):
        self.gene_list = gene_list
        self.disease_name = disease_name
        self.page_size = page_size
        self._base_url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

    def fetch_articles(self) -> List[Dict]:
        gene_wise_articles = []
        for gene in self.gene_list:
            query = f"{gene} AND {self.disease_name}"
            try:
                response = requests.get(self._base_url, params={
                    "query": query,
                    "format": "json",
                    "pageSize": self.page_size
                })
            except Exception as e:
                print(f"Error for gene: {gene} -> {e}")
                continue
            articles = []
            for i in response.json().get("resultList", {}).get("result", []):
                if "pmcid" in i:
                    articles.append({
                        "pmcid": i["pmcid"],
                        "pmid": i.get("pmid", ""),
                        "title": i.get("title", ""),
                        "source": i.get("source", ""),
                        "journal": i.get("journalTitle", "")
                    })
            gene_wise_articles.append({"gene":gene, "articles":articles})
        return gene_wise_articles


In [19]:
class FullTextParser:
    """Extract all human-readable text recursively from XML, ignoring only known garbage tags."""

    def __init__(self):
        self.rejected_tags = {
            "ref-list", "ref", "table", "table-wrap-foot", "license", 
            "supplementary-material", "ack", "app", "permissions"
        }

    def parse_fulltext(self, pmcid: str) -> List[str]:
        url = f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML"
        response = requests.get(url, params={"format": "xml"})
        if not response.ok:
            print(f"[Warning] Could not fetch fulltext for PMCID {pmcid}")
            return []

        try:
            root = ET.fromstring(response.content)
            text_blocks = []

            for elem in root.iter():
                tag = elem.tag.lower().split('}')[-1]  # remove namespace
                if tag in self.rejected_tags:
                    continue
                raw = ''.join(elem.itertext()).strip()
                cleaned = re.sub(r'\s+', ' ', raw)
                if cleaned:
                    text_blocks.append(cleaned)

            return text_blocks

        except ET.ParseError:
            print(f"[Error] Failed to parse XML for PMCID {pmcid}")
            return []


In [20]:
class EvidenceExtractor:
    def __init__(
        self,
        model_name: str = "ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli",
        hypotheses: List[str] = None
    ):
        self._tokenizer = AutoTokenizer.from_pretrained(model_name)
        self._model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self._model.eval()
        self._device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._model.to(self._device)
        self.nlp = spacy.load("en_core_web_sm")
        self.hypotheses = hypotheses or [
            "{gene} is associated with {disease}",
            "{gene} plays a role in {disease}",
            "{gene} is involved in the development of {disease}",
            "{gene} is a risk factor for {disease}"
        ]

    def _extract_sentences(self, text: str) -> List[str]:
        return [sent.text.strip() for sent in self.nlp(text).sents if sent.text.strip()]

    def _nli_infer(self, sentence: str, gene: str, disease: str) -> Optional[Dict[str, Any]]:
        for template in self.hypotheses:
            hypothesis = template.format(gene=gene, disease=disease)
            inputs = self._tokenizer(sentence, hypothesis, return_tensors="pt", truncation=True, padding=True).to(self._device)

            with torch.no_grad():
                outputs = self._model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                predicted = torch.argmax(probs, dim=1).item()

            label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
            if label_map[predicted] == "entailment" and probs[0][predicted] > 0.8:
                return {
                    "sentence": sentence,
                    "hypothesis": hypothesis,
                    "relation_label": label_map[predicted],
                    "score": float(probs[0][predicted])
                }
        return None

    def extract_from_text(self, text: str, gene: str, disease: str, pmcid: str = "") -> List[Dict[str, Any]]:
        evidence = []
        for sentence in self._extract_sentences(text):
            if gene.lower() in sentence.lower() and disease.lower() in sentence.lower():
                result = self._nli_infer(sentence, gene, disease)
                if result:
                    evidence.append({
                        "gene": gene,
                        "disease": disease,
                        "pmcid": pmcid,
                        **result
                    })
        return evidence


In [22]:
class PipelineRunner:
    def __init__(self, disease_name: str, gene_list: List[str], num_workers: int = 8):
        self.disease_name = disease_name
        self.gene_list = gene_list
        self.num_workers = num_workers
        self._fetcher = LiteratureFetcher(gene_list, disease_name)
        self._parser = FullTextParser()
        self._evidence_extractor = EvidenceExtractor()

    def _process_gene_articles(self, entry: Dict[str, Any]) -> List[Dict[str, Any]]:
        gene = entry["gene"]
        all_evidence = []
        for article in entry["articles"]:
            pmcid = article["pmcid"]
            try:
                text_blocks = self._parser.parse_fulltext(pmcid)
                if not text_blocks:
                    print(f"[Warning] No fulltext found for {pmcid}")
                    continue

                full_text = " ".join(text_blocks)
                evidence = self._evidence_extractor.extract_from_text(
                    full_text, gene, self.disease_name, pmcid
                )
                if evidence:
                    print(f"✅ Evidence found in {pmcid} (sentences: {len(evidence)})")
                    all_evidence.extend(evidence)
                else:
                    print(f"[Info] No relation found in {pmcid}")
            except Exception as e:
                print(f"[Error] Failed to process {pmcid}: {e}")
            time.sleep(0.3)  # Respectful API pacing
        return all_evidence

    def run(self) -> List[Dict[str, Any]]:
        all_gene_articles = self._fetcher.fetch_articles()

        with ThreadPool(self.num_workers) as pool:
            results = list(tqdm(
                pool.imap(self._process_gene_articles, all_gene_articles),
                total=len(all_gene_articles),
                desc="🔍 Processing Genes in Literature"
            ))

        return [item for sublist in results for item in sublist]

In [25]:
runner = PipelineRunner("Parkinson's disease", final_genes_scored.head(25)["symbol_x"].to_list(), num_workers=9)
results = runner.run()

Some weights of the model checkpoint at ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[Info] No relation found in PMC12178584
[Info] No relation found in PMC12086457
[Info] No relation found in PMC12033924
[Info] No relation found in PMC11775090
[Info] No relation found in PMC12095958
[Info] No relation found in PMC12211261
[Info] No relation found in PMC11971801
✅ Evidence found in PMC12106619 (sentences: 5)
[Info] No relation found in PMC12010390
[Info] No relation found in PMC10567903
[Info] No relation found in PMC12127898
✅ Evidence found in PMC12148662 (sentences: 8)
[Info] No relation found in PMC10801874
[Info] No relation found in PMC11971801
[Info] No relation found in PMC11662730
[Info] No relation found in PMC10324346
[Info] No relation found in PMC12127898
[Info] No relation found in PMC12084004
[Info] No relation found in PMC12073980
[Info] No relation found in PMC12211261
[Info] No relation found in PMC11662730
[Info] No relation found in PMC12122992
[Info] No relation found in PMC11505506
[Info] No relation found in PMC10363635
[Info] No relation found i

[Info] No relation found in PMC11014377
[Info] No relation found in PMC9696982
[Info] No relation found in PMC11992592
✅ Evidence found in PMC12242855 (sentences: 5)


[Info] No relation found in PMC12178584
[Info] No relation found in PMC9339187
[Info] No relation found in PMC10695175
[Info] No relation found in PMC9450364
[Info] No relation found in PMC12089919
[Info] No relation found in PMC7891830
[Info] No relation found in PMC11998164
[Info] No relation found in PMC8487122



🔍 Processing Genes in Literature:  20%|██        | 5/25 [22:14<48:07, 144.39s/it]  

...

✅ Evidence found in PMC11000071 (sentences: 14)
[Info] No relation found in PMC12071818
[Info] No relation found in PMC8721901
[Info] No relation found in PMC12027003
[Info] No relation found in PMC12010065
[Info] No relation found in PMC11769848
[Info] No relation found in PMC9395532
✅ Evidence found in PMC10667838 (sentences: 26)
[Info] No relation found in PMC8221893
[Info] No relation found in PMC11968493
[Info] No relation found in PMC12250488
[Info] No relation found in PMC11968493
[Info] No relation found in PMC10227369
[Warning] Could not fetch fulltext for PMCID PMC8028609
[Warning] No fulltext found for PMC8028609
[Info] No relation found in PMC11968493
[Warning] Could not fetch fulltext for PMCID PMC12205381
[Warning] No fulltext found for PMC12205381
[Info] No relation found in PMC11045791
[Info] No relation found in PMC12072996
[Info] No relation found in PMC9558701
✅ Evidence found in PMC12254814 (sentences: 27)
[Info] No relation found in PMC11354291
[Info] No relation f

[Info] No relation found in PMC12163413
[Info] No relation found in PMC11889530
✅ Evidence found in PMC11316225 (sentences: 30)
[Info] No relation found in PMC12086457
[Info] No relation found in PMC8192842
[Info] No relation found in PMC11928529
[Info] No relation found in PMC10695175
[Info] No relation found in PMC8268593
[Info] No relation found in PMC12072996
[Info] No relation found in PMC10770730
[Info] No relation found in PMC10097096
[Info] No relation found in PMC11735587
[Info] No relation found in PMC10878536
[Info] No relation found in PMC12098209
[Info] No relation found in PMC10816150
[Info] No relation found in PMC7511865
✅ Evidence found in PMC10630267 (sentences: 9)
[Info] No relation found in PMC12178584
[Info] No relation found in PMC5596942
[Info] No relation found in PMC11576416
✅ Evidence found in PMC10484682 (sentences: 11)
[Info] No relation found in PMC11831814
[Info] No relation found in PMC3930576
[Info] No relation found in PMC12208970


✅ Evidence found in PMC11094647 (sentences: 4)
[Info] No relation found in PMC7960542
✅ Evidence found in PMC11089466 (sentences: 4)
[Info] No relation found in PMC12178584


[Info] No relation found in PMC9809048
[Warning] Could not fetch fulltext for PMCID PMC8028609
[Warning] No fulltext found for PMC8028609
[Info] No relation found in PMC5465468
[Info] No relation found in PMC5733060
[Warning] Could not fetch fulltext for PMCID PMC6553931
[Warning] No fulltext found for PMC6553931
[Info] No relation found in PMC9576445
[Info] No relation found in PMC5062749
[Info] No relation found in PMC3825669


🔍 Processing Genes in Literature: 100%|██████████| 25/25 [45:25<00:00, 109.01s/it]


In [26]:
results

[{'gene': 'SNCA',
  'disease': "Parkinson's disease",
  'pmcid': 'PMC7902914',
  'sentence': "Frontiers in Neurology1664-2295Frontiers Media S.A.790291410.3389/fneur.2020.620585NeurologyBrief Research ReportAssociation of SNCA Parkinson's Disease Risk Polymorphisms With Disease Progression in Newly Diagnosed PatientsSzwedoAleksandra A.12†PedersenCamilla",
  'hypothesis': "SNCA is associated with Parkinson's disease",
  'relation_label': 'entailment',
  'score': 0.9997630715370178},
 {'gene': 'SNCA',
  'disease': "Parkinson's disease",
  'pmcid': 'PMC7902914',
  'sentence': "Objectives: To evaluate the impact of SNCA polymorphisms originally identified as risk factors for Parkinson's disease (PD) on the clinical presentation and progression of the disease in a large cohort of population-based patients with incident PD.Methods: Four hundred thirty-three patients and 417 controls from three longitudinal cohorts were included in the study.",
  'hypothesis': "SNCA is associated with Parkins